In [81]:
import os
import json
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal

from dotenv import load_dotenv
load_dotenv()

@dataclass(frozen=True)
class Provider:
    """One provider described as pure data"""
    
    name: str
    """Name of the provider"""
    env_var: str
    """Environment variable name for the provider's API key"""
    is_free: bool
    """Whether the provider is free or not"""
    base_url: str|None
    """Base URL for the provider's API"""
    model: str|None
    """Model name for the provider's API"""
    
    
PROVIDERS=[
    Provider(
        name="Groq",
        env_var="GROQ_API_KEY",
        is_free=False,
        base_url="https://api.groq.com/",
        model="openai/gpt-oss-120b"  
    ),
    # can i use one more geoq provider -> answer: yes, you can add another provider by creating a new instance of the Provider class and adding it to the PROVIDERS dictionary. For example:
    Provider(
        name="Groq2",
        env_var="GROQ2_API_KEY",
        is_free=False,
        base_url="https://api.groq.com/",
        model="openai/gpt-oss-20b"  
    ) 
]

def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var): # check if the environment variable for the provider's API key is set
            return provider
        
        expected = ", ".join(provider.env_var for provider in PROVIDERS)
        raise RuntimeError(f"No provider key set. Please set one of the following environment variables: {expected}")
        
def build_client(provider: Provider):
    from groq import Groq
    
    api_key = os.environ[provider.env_var]
    if provider.base_url is None:
        return Groq(api_key=api_key)
    else:
        return Groq(api_key=api_key, base_url=provider.base_url)
    
def have_any_key()->bool:
    return any(os.environ.get(provider.env_var) for provider in PROVIDERS)

def llm_reply(prompt: str, *, max_tokens: int = 400)->str:
    # send one user prompt...returns the assistant's text
    provider = select_provider()
    client = build_client(provider)
    response = client.chat.completions.create(
        model=provider.model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content 

In [82]:
WEATHER_DB = {
    "london": {"celsius": 22, "sky": "cloudy"},
    "paris": {"celsius": 24, "sky": "sunny"},
    "tallinn": {"celsius": 28, "sky": "sunny"},
    "moscow": {"celsius": 18, "sky": "rainy"},
    "tokyo": {"celsius": 30, "sky": "sunny"},
    "beijing": {"celsius": 26, "sky": "cloudy"},
    "new york": {"celsius": 28, "sky": "sunny"},
    "mumbai": {"celsius": 32, "sky": "rainy"},
    "cape town": {"celsius": 16, "sky": "cloudy"},
    "sydney": {"celsius": 20, "sky": "sunny"},
    "rio de janeiro": {"celsius": 22, "sky": "cloudy"},
    "cairo": {"celsius": 24, "sky": "sunny"},
    "mexico city": {"celsius": 20, "sky": "cloudy"},
}

In [83]:
def lookup_weather(location: str)->str:
    record = WEATHER_DB.get(location.lower())
    if record is None:
        return f"Sorry, I don't know the weather in {location}"
    return f"The weather in {location} is {record['celsius']} C and {record['sky']}."
    

In [84]:
# what the tool is capable of and how to use the tool
# this schema will be given to LLM
weather_schema = {
    "type":"function",
    "function":{
        "name":"lookup_weather", 
        "description":"Look up the weather for a location.",
        "parameters":{
            "type":"object",
            "properties":{
                "location":{"type":"string","description":"The location to look up the waether for."}
            },
            "required":["location"]
        }
    }
}

TOOLS = {
    "lookup_weather":lookup_weather
    # Other tools can be define
}

In [91]:

def ask_llm_with_tool(prompt: str, *, max_tokens: int = 400) -> str:
    import json

    provider = select_provider()
    client = build_client(provider)

    messages = [{"role": "user", "content": prompt}]

    # First LLM call
    response = client.chat.completions.create(
        model=provider.model,
        messages=messages,
        max_tokens=max_tokens,
        tools=[weather_schema],
        tool_choice="auto"
    )

    msg = response.choices[0].message

    # Print tool-call information
    if msg.tool_calls:
        print("Tool call detected")
        print("Total tool calls:", len(msg.tool_calls))

        messages.append(msg)

        # Execute each requested tool
        for tool_call in msg.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            print("Tool call name:", function_name)
            print("Tool call arguments:", tool_call.function.arguments)
            print("-" * 100)

            if function_name not in TOOLS:
                raise ValueError(f"Unknown tool: {function_name}")

            result = TOOLS[function_name](**arguments)
            print("Tool result:", result)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, default=str)
            })

        # Second LLM call: generate final answer
        response = client.chat.completions.create(
            model=provider.model,
            messages=messages,
            max_tokens=max_tokens
        )

    else:
        print("No tool call detected")

    final_message = response.choices[0].message.content or ""
    return final_message




In [92]:
message1 = ask_llm_with_tool("What is the weather in Tokyo")
print("Final response from llm:", message1)



Tool call detected
Total tool calls: 1
Tool call name: lookup_weather
Tool call arguments: {"location":"Tokyo"}
----------------------------------------------------------------------------------------------------
Tool result: The weather in Tokyo is 30 C and sunny.
Final response from llm: The current weather in Tokyo is **30 °C** with sunny skies.


In [87]:
message2 = llm_reply("What is the weather in Tokyo")
print("Final response from llm:", message2)


Final response from llm: I’m not able to pull in live weather data, so I can’t tell you the current conditions in Tokyo right now.  

For up‑to‑date information, you can check a reliable weather service such as:

- **Weather.com** – https://weather.com/weather/today/l/JPXX0085:1:JP  
- **AccuWeather** – https://www.accuweather.com/en/jp/tokyo/226396/weather-forecast/226396  
- **Japan Meteorological Agency** – https://www.jma.go.jp/en/forecast/  
- A smartphone weather app (e.g., the built‑in iOS Weather app, Google Weather, or any reputable third‑party app)

If you’re interested in what Tokyo’s weather is typically like throughout the year, here’s a quick overview:

| Season | Typical Temperatures (°C) | Common Weather |
|--------|---------------------------|----------------|
| **Winter (Dec‑Feb)** | 2 – 10 | Mostly dry and cool; occasional light snow |
| **Spring (Mar‑May)** | 10 – 20 | Mild, with cherry‑blossom season in late March‑early April |
| **Summer (Jun‑Aug)** | 22 – 31 | Ho